# GROUP BY & HAVING

In [30]:
import psycopg
import pandas as pd
conn = psycopg.connect(
    host = 'localhost',
    port= 5432,
    dbname ='chinook',
    user = 'postgres',
    password = '231165'
)

cur = conn.cursor()

## Introduction to GROUP BY

In [31]:
# Count tracks in each album


result = pd.read_sql('''
SELECT
    album_id,
    COUNT(*) AS track_count
FROM track
GROUP BY album_id
ORDER BY track_count DESC;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\1193663189.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,album_id,track_count
0,141,57
1,23,34
2,73,30
3,229,26
4,251,25
...,...,...
342,273,1
343,274,1
344,275,1
345,276,1


In [32]:
# Count invoices for each customer


result = pd.read_sql('''
SELECT
    customer_id,
    COUNT(*) AS invoice_count
FROM invoice
GROUP BY customer_id
ORDER BY invoice_count DESC;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\1102299986.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,invoice_count
0,8,7
1,54,7
2,4,7
3,34,7
4,51,7
5,52,7
6,10,7
7,35,7
8,45,7
9,6,7


## GROUP BY eith Multiple Agrregate Functions

In [33]:
# Summarize purchases for each customer

result = pd.read_sql('''
SELECT
    customer_id,
    COUNT(*) AS invoice_count,
    SUM(total) AS total_spent,
    AVG(total) AS average_invoice
FROM invoice
GROUP BY customer_id
ORDER BY total_spent DESC;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\4173116419.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,invoice_count,total_spent,average_invoice
0,6,7,49.62,7.088571
1,26,7,47.62,6.802857
2,57,7,46.62,6.660000
3,45,7,45.62,6.517143
4,46,7,45.62,6.517143
5,28,7,43.62,6.231429
6,37,7,43.62,6.231429
7,24,7,43.62,6.231429
8,7,7,42.62,6.088571
9,25,7,42.62,6.088571


In [34]:
# Count invoices per customer and year

result = pd.read_sql('''
SELECT
    customer_id,
    EXTRACT(YEAR FROM invoice_date) AS invoice_year,
    COUNT(*) AS invoice_count
FROM invoice
GROUP BY
    customer_id,
    EXTRACT(YEAR FROM invoice_date)
ORDER BY
    customer_id,
    invoice_year;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\1723172622.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,invoice_year,invoice_count
0,1,2022.0,3
1,1,2023.0,1
2,1,2024.0,2
3,1,2025.0,1
4,2,2021.0,3
...,...,...,...
227,58,2025.0,3
228,59,2021.0,2
229,59,2022.0,1
230,59,2023.0,2


## Introduction to HAVING

In [35]:
# Find customers with more than five invoices

result = pd.read_sql('''
SELECT
    customer_id,
    COUNT(*) AS invoice_count
FROM invoice
GROUP BY customer_id
HAVING COUNT(*) > 5
ORDER BY invoice_count DESC;
''', conn)

result


C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\4283661853.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,invoice_count
0,8,7
1,54,7
2,4,7
3,34,7
4,51,7
5,52,7
6,10,7
7,35,7
8,45,7
9,6,7


In [36]:
# Find customers with total invoices above 40

result = pd.read_sql('''
SELECT
    customer_id,
    SUM(total) AS total_spent
FROM invoice
GROUP BY customer_id
HAVING SUM(total) > 40
ORDER BY total_spent DESC;
''', conn)

result

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\2678048715.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,total_spent
0,6,49.62
1,26,47.62
2,57,46.62
3,45,45.62
4,46,45.62
5,28,43.62
6,37,43.62
7,24,43.62
8,25,42.62
9,7,42.62


## WHERE vs HAVING

In [37]:
result = pd.read_sql('''
SELECT
    customer_id,
    SUM(total) AS total_spent
FROM invoice
WHERE EXTRACT(YEAR FROM invoice_date) = '2023'
GROUP BY customer_id
HAVING SUM(total) > 10
ORDER BY total_spent DESC;
''', conn)

result

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\3407433881.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,customer_id,total_spent
0,46,32.75
1,29,24.75
2,8,24.75
3,50,22.77
4,25,20.84
5,42,17.84
6,4,17.84
7,44,15.88
8,17,15.84
9,59,15.84


## PostgreSQL vs SQLite

In [38]:
# Extract the year from a date ===> PostgreSQL

result = pd.read_sql('''
SELECT DISTINCT EXTRACT(YEAR FROM invoice_date)
FROM invoice
''', conn)

result

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_364\3575720577.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql('''


,extract
0,2024.0
1,2023.0
2,2022.0
3,2025.0
4,2021.0
